# 📊 Fund Performance Analytics — Bluestock MF Capstone
## Day 4: Performance Metrics

**Metrics Computed:** Daily Returns | CAGR | Sharpe | Sortino | Alpha | Beta | Max Drawdown | Scorecard  
**Rf (Risk-Free Rate):** 6.5% (RBI Repo Rate Proxy) | **Trading Days:** 252  

---


## ⚙️ Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path
from IPython.display import Image, display

BASE   = Path().resolve().parent
RAW    = BASE / "data" / "raw"
PROC   = BASE / "data" / "processed"
CHARTS = BASE / "reports" / "charts"

RF_ANNUAL    = 0.065
RF_DAILY     = RF_ANNUAL / 252
TRADING_DAYS = 252

COLORS = ["#6c8ebf","#82b366","#d6a91a","#e07b54","#9b72cf",
          "#4db6ac","#e57373","#81c784","#ffb74d","#64b5f6"]

plt.rcParams.update({
    "figure.facecolor":"#0f1117","axes.facecolor":"#1a1d2e",
    "axes.labelcolor":"#ccccdd","xtick.color":"#aaaacc",
    "ytick.color":"#aaaacc","text.color":"#ccccdd",
    "grid.color":"#2a2d3e","grid.alpha":0.5,
})

nav   = pd.read_csv(RAW/"02_nav_history.csv", parse_dates=["date"])
funds = pd.read_csv(RAW/"01_fund_master.csv")
bench = pd.read_csv(RAW/"10_benchmark_indices.csv", parse_dates=["date"])
nav   = nav.merge(funds[["amfi_code","scheme_name","fund_house","sub_category","plan","expense_ratio_pct"]], on="amfi_code", how="left")
nav   = nav.sort_values(["amfi_code","date"]).reset_index(drop=True)

print(f"NAV rows: {len(nav):,} | Funds: {nav['amfi_code'].nunique()} | Date range: {nav['date'].min().date()} → {nav['date'].max().date()}")
print("✅ Setup complete!")

---
## 📈 Task 1: Daily Returns
**Formula:** `daily_return = (NAV_t / NAV_t-1) - 1`


In [ ]:
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()
stats = nav["daily_return"].dropna().describe()

print("Daily Return Distribution:")
print(f"  Mean   : {stats['mean']*100:.4f}%")
print(f"  Std    : {stats['std']*100:.4f}%")
print(f"  Min    : {stats['min']*100:.4f}%")
print(f"  Max    : {stats['max']*100:.4f}%")
print(f"  Outliers >15% : {(nav['daily_return'].abs() > 0.15).sum()} ✅")

fig, ax = plt.subplots(figsize=(12,5))
fig.patch.set_facecolor("#0f1117")
ax.hist(nav["daily_return"].dropna()*100, bins=100, color="#6c8ebf", alpha=0.85, edgecolor="#1a1d2e")
ax.axvline(0, color="#ffb74d", linestyle="--", linewidth=1.5)
ax.set_title("Distribution of Daily Returns — All 40 Funds")
ax.set_xlabel("Daily Return (%)"); ax.set_ylabel("Frequency")
ax.grid(True); plt.tight_layout(); plt.show()

---
## 📊 Task 2: CAGR (1yr, 3yr, 5yr)
**Formula:** `CAGR = (NAV_end / NAV_start) ^ (1/n) - 1`


In [ ]:
latest_date = nav["date"].max()
cagr_results = []

for code, grp in nav.groupby("amfi_code"):
    grp = grp.sort_values("date")
    row = {"amfi_code": code,
           "scheme_name": grp["scheme_name"].iloc[0],
           "fund_house": grp["fund_house"].iloc[0],
           "sub_category": grp["sub_category"].iloc[0],
           "plan": grp["plan"].iloc[0]}
    for years, label in [(1,"cagr_1yr"),(3,"cagr_3yr"),(5,"cagr_5yr")]:
        sub = grp[grp["date"] >= latest_date - pd.DateOffset(years=years)]
        if len(sub) >= 50:
            n = (sub["date"].iloc[-1] - sub["date"].iloc[0]).days / 365.25
            row[label] = round(((sub["nav"].iloc[-1]/sub["nav"].iloc[0])**(1/n)-1)*100, 4)
        else:
            row[label] = np.nan
    cagr_results.append(row)

cagr_df = pd.DataFrame(cagr_results).sort_values("cagr_3yr", ascending=False)
print(f"Average 1yr CAGR: {cagr_df['cagr_1yr'].mean():.2f}%")
print(f"Average 3yr CAGR: {cagr_df['cagr_3yr'].mean():.2f}%")
print(f"Average 5yr CAGR: {cagr_df['cagr_5yr'].mean():.2f}%")
cagr_df[["scheme_name","sub_category","plan","cagr_1yr","cagr_3yr","cagr_5yr"]].head(10)

In [ ]:
# CAGR Comparison Chart
fig = px.bar(cagr_df.head(15), x="cagr_3yr",
             y=[n.split("-")[0].strip()[:20] for n in cagr_df.head(15)["scheme_name"]],
             orientation="h", color="sub_category", title="Top 15 Funds — 3-Year CAGR (%)",
             labels={"x":"3-Year CAGR (%)","y":"Fund"}, template="plotly_dark", height=550)
fig.show()

---
## ⚡ Task 3: Sharpe Ratio
**Formula:** `Sharpe = (Rp - Rf) / Std(Rp) × √252`  
**Rf = 6.5% annual (RBI Repo Rate)**


In [ ]:
sharpe_results = []
for code, grp in nav.groupby("amfi_code"):
    ret = grp["daily_return"].dropna()
    if len(ret) < 50: continue
    excess = ret - RF_DAILY
    sharpe = (excess.mean() / ret.std()) * np.sqrt(TRADING_DAYS)
    sharpe_results.append({
        "amfi_code": code,
        "scheme_name": grp["scheme_name"].iloc[0],
        "fund_house": grp["fund_house"].iloc[0],
        "sub_category": grp["sub_category"].iloc[0],
        "sharpe_ratio": round(sharpe, 4),
        "ann_return_pct": round(ret.mean()*TRADING_DAYS*100, 4),
        "ann_vol_pct": round(ret.std()*np.sqrt(TRADING_DAYS)*100, 4),
    })

sharpe_df = pd.DataFrame(sharpe_results).sort_values("sharpe_ratio", ascending=False)
sharpe_df["sharpe_rank"] = range(1, len(sharpe_df)+1)
print(f"Best  Sharpe: {sharpe_df.iloc[0]['sharpe_ratio']:.4f} — {sharpe_df.iloc[0]['scheme_name'][:40]}")
print(f"Worst Sharpe: {sharpe_df.iloc[-1]['sharpe_ratio']:.4f} — {sharpe_df.iloc[-1]['scheme_name'][:40]}")
sharpe_df[["scheme_name","sub_category","sharpe_ratio","ann_return_pct","ann_vol_pct","sharpe_rank"]].head(10)

---
## 📉 Task 4: Sortino Ratio
**Formula:** `Sortino = (Rp - Rf) × √252 / Downside_Std`  
Uses only **negative return days** in denominator.


In [ ]:
sortino_results = []
for code, grp in nav.groupby("amfi_code"):
    ret = grp["daily_return"].dropna()
    if len(ret) < 50: continue
    downside = ret[ret < 0]
    down_std = downside.std() * np.sqrt(TRADING_DAYS)
    sortino  = ((ret - RF_DAILY).mean() * TRADING_DAYS) / down_std if down_std > 0 else np.nan
    sortino_results.append({
        "amfi_code": code,
        "scheme_name": grp["scheme_name"].iloc[0],
        "sub_category": grp["sub_category"].iloc[0],
        "sortino_ratio": round(sortino, 4),
        "downside_vol_pct": round(downside.std()*np.sqrt(TRADING_DAYS)*100, 4),
    })

sortino_df = pd.DataFrame(sortino_results).sort_values("sortino_ratio", ascending=False)
sortino_df["sortino_rank"] = range(1, len(sortino_df)+1)
print(f"Avg Sortino: {sortino_df['sortino_ratio'].mean():.4f}")
sortino_df[["scheme_name","sub_category","sortino_ratio","downside_vol_pct","sortino_rank"]].head(10)

---
## 🔬 Task 5: Alpha & Beta (OLS Regression)
**Method:** `scipy.stats.linregress(bench_return, fund_return)`  
**Alpha (annualised):** `intercept × 252`  
**Beta:** `slope` (sensitivity to NIFTY 100)


In [ ]:
from scipy import stats as sp_stats

nifty100 = bench[bench["index_name"]=="NIFTY100"].sort_values("date").copy()
nifty100["bench_return"] = nifty100["close_value"].pct_change()
nifty100 = nifty100[["date","bench_return"]].dropna()

alpha_results = []
for code, grp in nav.groupby("amfi_code"):
    grp = grp.sort_values("date").dropna(subset=["daily_return"])
    merged = grp.merge(nifty100, on="date", how="inner")
    if len(merged) < 100: continue
    slope, intercept, r_val, p_val, _ = sp_stats.linregress(
        merged["bench_return"], merged["daily_return"])
    alpha_results.append({
        "amfi_code": code,
        "scheme_name": grp["scheme_name"].iloc[0],
        "fund_house": grp["fund_house"].iloc[0],
        "sub_category": grp["sub_category"].iloc[0],
        "plan": grp["plan"].iloc[0],
        "alpha_annual_pct": round(intercept * TRADING_DAYS * 100, 4),
        "beta": round(slope, 4),
        "r_squared": round(r_val**2, 4),
    })

alpha_df = pd.DataFrame(alpha_results).sort_values("alpha_annual_pct", ascending=False)
alpha_df["alpha_rank"] = range(1, len(alpha_df)+1)
print(f"Avg Beta  : {alpha_df['beta'].mean():.4f}")
print(f"Avg Alpha : {alpha_df['alpha_annual_pct'].mean():.4f}%")
alpha_df[["scheme_name","sub_category","alpha_annual_pct","beta","r_squared"]].head(10)

In [ ]:
# Alpha vs Beta scatter
fig = px.scatter(alpha_df, x="beta", y="alpha_annual_pct",
    color="sub_category", hover_name="scheme_name",
    text=alpha_df["fund_house"].str.replace(" Mutual Fund","").str.replace(" MF",""),
    title="Alpha vs Beta — All 40 Funds (vs NIFTY 100)",
    labels={"beta":"Beta","alpha_annual_pct":"Alpha (Annual %)"},
    template="plotly_dark", height=500)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.add_vline(x=1, line_dash="dash", line_color="gray")
fig.show()

---
## 📉 Task 6: Maximum Drawdown
**Formula:** `max_dd = min(NAV / cummax(NAV) - 1)`


In [ ]:
dd_results = []
for code, grp in nav.groupby("amfi_code"):
    grp = grp.sort_values("date").copy()
    grp["rolling_max"] = grp["nav"].cummax()
    grp["drawdown"]    = grp["nav"] / grp["rolling_max"] - 1
    max_dd_idx  = grp["drawdown"].idxmin()
    max_dd_date = grp.loc[max_dd_idx, "date"]
    peak_idx    = grp.loc[:max_dd_idx, "nav"].idxmax()
    peak_date   = grp.loc[peak_idx, "date"]
    dd_results.append({
        "amfi_code": code,
        "scheme_name": grp["scheme_name"].iloc[0],
        "sub_category": grp["sub_category"].iloc[0],
        "max_drawdown_pct": round(grp["drawdown"].min()*100, 4),
        "drawdown_peak_date": str(peak_date.date()),
        "drawdown_trough_date": str(max_dd_date.date()),
        "drawdown_duration_days": (max_dd_date - peak_date).days,
    })

dd_df = pd.DataFrame(dd_results).sort_values("max_drawdown_pct")
dd_df["dd_rank"] = range(1, len(dd_df)+1)
print(f"Worst Drawdown: {dd_df.iloc[0]['max_drawdown_pct']:.2f}% — {dd_df.iloc[0]['scheme_name'][:40]}")
print(f"Best  Drawdown: {dd_df.iloc[-1]['max_drawdown_pct']:.2f}% — {dd_df.iloc[-1]['scheme_name'][:40]}")
dd_df[["scheme_name","sub_category","max_drawdown_pct","drawdown_peak_date","drawdown_trough_date","drawdown_duration_days"]].head(10)

---
## 🏆 Task 7: Fund Scorecard (0–100)
**Weights:**
- 30% × 3yr CAGR rank
- 25% × Sharpe rank  
- 20% × Alpha rank
- 15% × Expense Ratio rank (inverse — lower is better)
- 10% × Max Drawdown rank (inverse — less negative is better)


In [ ]:
sc = cagr_df[["amfi_code","scheme_name","fund_house","sub_category","plan","cagr_3yr"]].copy()
sc = sc.merge(sharpe_df[["amfi_code","sharpe_ratio","sharpe_rank"]], on="amfi_code", how="left")
sc = sc.merge(alpha_df[["amfi_code","alpha_annual_pct","alpha_rank"]], on="amfi_code", how="left")
sc = sc.merge(dd_df[["amfi_code","max_drawdown_pct","dd_rank"]], on="amfi_code", how="left")
sc = sc.merge(funds[["amfi_code","expense_ratio_pct"]], on="amfi_code", how="left")

n = len(sc)
sc["cagr_rank"] = sc["cagr_3yr"].rank(ascending=False)
sc["er_rank"]   = sc["expense_ratio_pct"].rank(ascending=True)
sc["dd_rank2"]  = sc["max_drawdown_pct"].rank(ascending=False)

sc["score"] = (
    0.30 * (1 - (sc["cagr_rank"]   - 1) / (n-1)) * 100 +
    0.25 * (1 - (sc["sharpe_rank"] - 1) / (n-1)) * 100 +
    0.20 * (1 - (sc["alpha_rank"]  - 1) / (n-1)) * 100 +
    0.15 * (1 - (sc["er_rank"]     - 1) / (n-1)) * 100 +
    0.10 * (1 - (sc["dd_rank2"]    - 1) / (n-1)) * 100
).round(2)
sc["overall_rank"] = sc["score"].rank(ascending=False).astype(int)
sc = sc.sort_values("overall_rank")

print("🏆 Top 10 Funds by Scorecard:")
sc[["overall_rank","scheme_name","sub_category","cagr_3yr","sharpe_ratio","alpha_annual_pct","score"]].head(10)

In [ ]:
# Scorecard chart
fig = px.bar(sc.head(15),
    x="score",
    y=[n.split("-")[0].strip()[:22] for n in sc.head(15)["scheme_name"]],
    orientation="h", color="sub_category",
    title="Fund Scorecard — Top 15 Funds (0–100)",
    labels={"x":"Score (0-100)","y":"Fund"},
    template="plotly_dark", height=550)
fig.update_xaxes(range=[0,100])
fig.show()

---
## 📉 Task 8: Benchmark Comparison
**Metrics:** Normalised performance vs NIFTY 50 & NIFTY 100 + Tracking Error


In [ ]:
display(Image(str(CHARTS/"D4_benchmark_comparison.png"), width=950))

In [ ]:
# Interactive benchmark chart
top5 = sc.head(5)["amfi_code"].tolist()
start = pd.Timestamp("2023-01-01")

fig = go.Figure()
for idx_name, color, dash in [("NIFTY50","#ffb74d","dash"),("NIFTY100","#ff8a65","dot")]:
    b = bench[bench["index_name"]==idx_name].sort_values("date")
    b = b[b["date"] >= start].copy()
    b["norm"] = b["close_value"] / b["close_value"].iloc[0] * 100
    fig.add_trace(go.Scatter(x=b["date"], y=b["norm"], name=idx_name,
        line=dict(color=color, width=2.5, dash=dash)))

for i, code in enumerate(top5):
    g = nav[nav["amfi_code"]==code].sort_values("date")
    g = g[g["date"] >= start].copy()
    if len(g) < 10: continue
    g["norm"] = g["nav"] / g["nav"].iloc[0] * 100
    name = g["scheme_name"].iloc[0].split("-")[0].strip()[:20]
    fig.add_trace(go.Scatter(x=g["date"], y=g["norm"], name=name,
        line=dict(color=COLORS[i], width=1.8)))

fig.update_layout(title="Top 5 Funds vs NIFTY 50 & NIFTY 100 (Base=100, Jan 2023)",
    xaxis_title="Date", yaxis_title="Normalised Value",
    template="plotly_dark", height=500)
fig.show()

---
## 💾 Save Deliverables

In [ ]:
# Save all output CSVs
sc.to_csv(PROC/"fund_scorecard.csv", index=False)
alpha_df.to_csv(PROC/"alpha_beta.csv", index=False)

full = (cagr_df
    .merge(sharpe_df[["amfi_code","sharpe_ratio","ann_vol_pct"]], on="amfi_code", how="left")
    .merge(sortino_df[["amfi_code","sortino_ratio"]], on="amfi_code", how="left")
    .merge(alpha_df[["amfi_code","alpha_annual_pct","beta","r_squared"]], on="amfi_code", how="left")
    .merge(dd_df[["amfi_code","max_drawdown_pct","drawdown_peak_date","drawdown_trough_date"]], on="amfi_code", how="left")
    .merge(sc[["amfi_code","score","overall_rank"]], on="amfi_code", how="left"))

full.to_csv(PROC/"full_performance_metrics.csv", index=False)
print("✅ fund_scorecard.csv saved!")
print("✅ alpha_beta.csv saved!")
print("✅ full_performance_metrics.csv saved!")
print(f"\n📊 Final Scorecard — Top 5:")
sc[["overall_rank","scheme_name","score"]].head(5).to_string(index=False)

---
## 📝 Key Performance Findings

| # | Finding |
|---|---------|
| 1 | Average 3yr CAGR across all 40 funds: **16.41%** |
| 2 | Best Sharpe Ratio: **1.45** (Mirae Asset Large Cap) |
| 3 | Avg Beta close to 1 — funds closely track market |
| 4 | Worst Max Drawdown: **-52.57%** (Small Cap during corrections) |
| 5 | Direct plans score higher on scorecard due to lower expense ratios |
| 6 | Mid Cap funds show highest Alpha but also highest volatility |
| 7 | Sortino > Sharpe for most funds — good upside capture |
| 8 | Top scorer: **Mirae Asset Large Cap Regular** with score **85.9/100** |
